# IDE Model Configuration — Self-Hosted Model

Configure your IDE to use the **self-hosted model** (qwen36-27b) served via RHOAI MaaS gateway. No external AI API keys required.

> **Prerequisites:**
> - MCP server registration → `../1_mcp_servers/4_connect_ide_clients.ipynb`
> - MaaS API key → `../2_maas/2_enable_maas.ipynb` (MAAS_API_KEY must be set in `.env`)

| IDE | Model Config Method | Requirement |
|-----|-------------------|-------------|
| **Cursor** | Settings → Models → OpenAI Compatible | Cursor installed |
| **VS Code** | BYOK Custom Endpoint (`chatLanguageModels.json`) | VS Code installed |
| **Claude Code** | Environment variables (`ANTHROPIC_BASE_URL`) | Claude Code CLI installed |

> **Self-signed certificate:** Run `launchctl setenv NODE_TLS_REJECT_UNAUTHORIZED 0` once (macOS) and restart your IDE. See `../1_mcp_servers/4_connect_ide_clients.ipynb` for details.

**Sections:**
1. Cursor — Model configuration
2. VS Code — Model configuration (BYOK)
3. Claude Code — Model configuration (env vars)
4. MaaS Gateway — Unified Endpoint
5. Verify connectivity

## 0. Load Environment

In [2]:
import subprocess, json, os
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config.openshift.io", "cluster",
                        "-o", "jsonpath={.spec.domain}"], capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
MODEL_NAME = os.getenv("MODEL_NAME", "qwen36-27b")
MODEL_ENDPOINT = os.getenv("MODEL_ENDPOINT", f"https://maas-api.{CLUSTER_DOMAIN}/{MODEL_NAMESPACE}/{MODEL_NAME}")
MAAS_API_KEY = os.getenv("MAAS_API_KEY", "")

if MAAS_API_KEY and len(MAAS_API_KEY) > 16:
    api_key_masked = MAAS_API_KEY[:12] + "..." + MAAS_API_KEY[-4:]
else:
    api_key_masked = "<YOUR_MAAS_API_KEY>"

print(f"Cluster:       {CLUSTER_DOMAIN}")
print(f"Model:         {MODEL_NAME}")
print(f"Model URL:     {MODEL_ENDPOINT}/v1")
print(f"MaaS API Key:  {api_key_masked}")

if not MAAS_API_KEY:
    print("")
    print("⚠️  MAAS_API_KEY is not set in .env")
    print("   Run ../2_maas/2_enable_maas.ipynb first to generate an API key.")

Cluster:       apps.openshift-cluster.sandbox1785.opentlc.com
Model:         qwen36-27b
Model URL:     https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b/v1
MaaS API Key:  sk-oai-1Bq6X...bk0h


---
## 1. Cursor IDE — Model Configuration

In Cursor: **Settings → Models → Add Model**

| Setting | Value |
|---------|-------|
| Provider | OpenAI Compatible |
| Base URL | `<MODEL_ENDPOINT>/v1` |
| Model Name | `<MODEL_NAME>` |
| API Key | MaaS API key (from `.env`) |

In [3]:
print("Cursor Model Settings:")
print("=" * 50)
print(f"  Provider:   OpenAI Compatible")
print(f"  Base URL:   {MODEL_ENDPOINT}/v1")
print(f"  Model Name: {MODEL_NAME}")
print(f"  API Key:    {api_key_masked}")
print("")
print("Steps:")
print("  1. Open Cursor Settings (Cmd+,)")
print("  2. Go to Models section")
print("  3. Click 'Add Model' → 'OpenAI Compatible'")
print("  4. Enter the values above")
print(f"  5. Select '{MODEL_NAME}' as your default model")

Cursor Model Settings:
  Provider:   OpenAI Compatible
  Base URL:   https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b/v1
  Model Name: qwen36-27b
  API Key:    sk-oai-1Bq6X...bk0h

Steps:
  1. Open Cursor Settings (Cmd+,)
  2. Go to Models section
  3. Click 'Add Model' → 'OpenAI Compatible'
  4. Enter the values above
  5. Select 'qwen36-27b' as your default model


---
## 2. VS Code — Model Configuration (BYOK)

Use **BYOK (Bring Your Own Key)** to register the self-hosted model. No Copilot subscription or GitHub login required.

1. Command Palette (`Cmd+Shift+P`) → `Chat: Manage Language Models`
2. Select `Custom Endpoint` → enter name and API key
3. The configuration is saved to `.vscode/chatLanguageModels.json`:

In [4]:
vscode_model_config = [
    {
        "name": "RHOAI",
        "vendor": "customendpoint",
        "apiKey": api_key_masked,
        "apiType": "chat-completions",
        "models": [
            {
                "id": MODEL_NAME,
                "name": f"{MODEL_NAME} (RHOAI)",
                "url": f"{MODEL_ENDPOINT}/v1/chat/completions",
                "toolCalling": True,
                "maxInputTokens": 32768,
                "maxOutputTokens": 4096
            }
        ]
    }
]

print("=== .vscode/chatLanguageModels.json ===")
print(json.dumps(vscode_model_config, indent=2))
print("")
print("Steps:")
print("  1. Cmd+Shift+P → 'Chat: Manage Language Models'")
print("  2. Select 'Custom Endpoint' → enter 'RHOAI' → paste API key")
print("  3. Configuration saved to chatLanguageModels.json")
print("  4. Select model from Chat model picker")
print("")
if not MAAS_API_KEY:
    print("NOTE: Replace API key with your actual MAAS_API_KEY.")
else:
    print(f"NOTE: apiKey is masked. Use the full MAAS_API_KEY from .env when configuring.")

=== .vscode/chatLanguageModels.json ===
[
  {
    "name": "RHOAI",
    "vendor": "customendpoint",
    "apiKey": "sk-oai-1Bq6X...bk0h",
    "apiType": "chat-completions",
    "models": [
      {
        "id": "qwen36-27b",
        "name": "qwen36-27b (RHOAI)",
        "url": "https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b/v1/chat/completions",
        "toolCalling": true,
        "maxInputTokens": 32768,
        "maxOutputTokens": 4096
      }
    ]
  }
]

Steps:
  1. Cmd+Shift+P → 'Chat: Manage Language Models'
  2. Select 'Custom Endpoint' → enter 'RHOAI' → paste API key
  3. Configuration saved to chatLanguageModels.json
  4. Select model from Chat model picker

NOTE: apiKey is masked. Use the full MAAS_API_KEY from .env when configuring.


---
## 3. Claude Code — Model Configuration

Claude Code connects to self-hosted models via environment variables.

| Env Var | Purpose |
|---------|--------|
| `ANTHROPIC_BASE_URL` | MaaS inference endpoint |
| `ANTHROPIC_AUTH_TOKEN` | MaaS API key (**NOT** `ANTHROPIC_API_KEY`) |
| `ANTHROPIC_DEFAULT_*_MODEL` | Model name for each tier (all same model) |
| `MAX_THINKING_TOKENS` | `0` — disable thinking (vLLM doesn't support it) |

> **Critical:** Use `ANTHROPIC_AUTH_TOKEN`, not `ANTHROPIC_API_KEY`.

In [5]:
claude_launch_cmd = f"""ANTHROPIC_BASE_URL="{MODEL_ENDPOINT}" \\
ANTHROPIC_AUTH_TOKEN="{api_key_masked}" \\
ANTHROPIC_DEFAULT_OPUS_MODEL="{MODEL_NAME}" \\
ANTHROPIC_DEFAULT_SONNET_MODEL="{MODEL_NAME}" \\
ANTHROPIC_DEFAULT_HAIKU_MODEL="{MODEL_NAME}" \\
CLAUDE_CODE_FILE_READ_MAX_OUTPUT_TOKENS="30000" \\
CLAUDE_CODE_MAX_OUTPUT_TOKENS="50000" \\
MAX_THINKING_TOKENS="0" \\
claude"""

print("=== Launch Claude Code with self-hosted model ===")
print("")
print(claude_launch_cmd)
print("")
print("Copy-paste the command above into your terminal.")
if not MAAS_API_KEY:
    print("NOTE: Replace the API key placeholder with your actual MAAS_API_KEY.")
else:
    print(f"NOTE: Key is masked. Use full MAAS_API_KEY from .env when copy-pasting.")

=== Launch Claude Code with self-hosted model ===

ANTHROPIC_BASE_URL="https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b" \
ANTHROPIC_AUTH_TOKEN="sk-oai-1Bq6X...bk0h" \
ANTHROPIC_DEFAULT_OPUS_MODEL="qwen36-27b" \
ANTHROPIC_DEFAULT_SONNET_MODEL="qwen36-27b" \
ANTHROPIC_DEFAULT_HAIKU_MODEL="qwen36-27b" \
CLAUDE_CODE_FILE_READ_MAX_OUTPUT_TOKENS="30000" \
CLAUDE_CODE_MAX_OUTPUT_TOKENS="50000" \
MAX_THINKING_TOKENS="0" \
claude

Copy-paste the command above into your terminal.
NOTE: Key is masked. Use full MAAS_API_KEY from .env when copy-pasting.


### Shell Script (Optional — `run-claude.sh`)

In [6]:
script_content = f"""#!/bin/bash
# Launch Claude Code with self-hosted RHOAI model
# Usage: source run-claude.sh

export ANTHROPIC_BASE_URL="{MODEL_ENDPOINT}"
export ANTHROPIC_AUTH_TOKEN="${{MAAS_API_KEY:-{api_key_masked}}}"
export ANTHROPIC_DEFAULT_OPUS_MODEL="{MODEL_NAME}"
export ANTHROPIC_DEFAULT_SONNET_MODEL="{MODEL_NAME}"
export ANTHROPIC_DEFAULT_HAIKU_MODEL="{MODEL_NAME}"
export CLAUDE_CODE_FILE_READ_MAX_OUTPUT_TOKENS="30000"
export CLAUDE_CODE_MAX_OUTPUT_TOKENS="50000"
export MAX_THINKING_TOKENS="0"

echo "Claude Code configured for: ${{ANTHROPIC_DEFAULT_SONNET_MODEL}}"
echo "Endpoint: ${{ANTHROPIC_BASE_URL}}"
claude "$@"
"""

print("=== run-claude.sh ===")
print(script_content)

=== run-claude.sh ===
#!/bin/bash
# Launch Claude Code with self-hosted RHOAI model
# Usage: source run-claude.sh

export ANTHROPIC_BASE_URL="https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/demo/qwen36-27b"
export ANTHROPIC_AUTH_TOKEN="${MAAS_API_KEY:-sk-oai-1Bq6X...bk0h}"
export ANTHROPIC_DEFAULT_OPUS_MODEL="qwen36-27b"
export ANTHROPIC_DEFAULT_SONNET_MODEL="qwen36-27b"
export ANTHROPIC_DEFAULT_HAIKU_MODEL="qwen36-27b"
export CLAUDE_CODE_FILE_READ_MAX_OUTPUT_TOKENS="30000"
export CLAUDE_CODE_MAX_OUTPUT_TOKENS="50000"
export MAX_THINKING_TOKENS="0"

echo "Claude Code configured for: ${ANTHROPIC_DEFAULT_SONNET_MODEL}"
echo "Endpoint: ${ANTHROPIC_BASE_URL}"
claude "$@"



---
## 4. MaaS Gateway — Unified Endpoint

The **MaaS Gateway** (configured in Phase 2) provides unified auth for both model and MCP access with a single API key.

| | Direct Route (Phase 1) | MaaS Gateway (Phase 2+) |
|-|--------------------------|-------------------------|
| MCP endpoints | 5 separate URLs | 1 unified URL |
| Authentication | None | API key |
| Rate limiting | None | Per-subscription token limits |
| Config entries | 5 MCP + 1 model | 1 MCP gateway + 1 model |

In [7]:
MAAS_GW = f"https://maas-api.{CLUSTER_DOMAIN}"

print("=== MaaS Gateway Mode ===")
print("")
print("Cursor (.cursor/mcp.json):")
cursor_maas = {
    "mcpServers": {
        "mcp-gateway": {
            "url": f"{MAAS_GW}/mcp/mcp",
            "headers": {"Authorization": f"Bearer {api_key_masked}"}
        }
    }
}
print(json.dumps(cursor_maas, indent=2))

print("")
print("VS Code (.vscode/mcp.json):")
vscode_maas = {
    "servers": {
        "mcp-gateway": {
            "type": "http",
            "url": f"{MAAS_GW}/mcp/mcp",
            "headers": {"Authorization": f"Bearer {api_key_masked}"}
        }
    }
}
print(json.dumps(vscode_maas, indent=2))

print("")
print("Claude Code (CLI):")
print(f"  claude mcp add --transport http \\")
print(f"    --header \"Authorization: Bearer {api_key_masked}\" \\")
print(f"    mcp-gateway \"{MAAS_GW}/mcp/mcp\"")

=== MaaS Gateway Mode ===

Cursor (.cursor/mcp.json):
{
  "mcpServers": {
    "mcp-gateway": {
      "url": "https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/mcp",
      "headers": {
        "Authorization": "Bearer sk-oai-1Bq6X...bk0h"
      }
    }
  }
}

VS Code (.vscode/mcp.json):
{
  "servers": {
    "mcp-gateway": {
      "type": "http",
      "url": "https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/mcp",
      "headers": {
        "Authorization": "Bearer sk-oai-1Bq6X...bk0h"
      }
    }
  }
}

Claude Code (CLI):
  claude mcp add --transport http \
    --header "Authorization: Bearer sk-oai-1Bq6X...bk0h" \
    mcp-gateway "https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com/mcp/mcp"


---
## 5. Verify Model Connectivity

In [8]:
import urllib.request, ssl

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

print("=== Model Endpoint Check ===")
print("")
try:
    model_url = f"{MODEL_ENDPOINT}/v1/models"
    headers = {"Content-Type": "application/json"}
    if MAAS_API_KEY:
        headers["Authorization"] = f"Bearer {MAAS_API_KEY}"
    req = urllib.request.Request(model_url, headers=headers)
    with urllib.request.urlopen(req, context=ctx, timeout=10) as resp:
        data = json.loads(resp.read())
        models = [m["id"] for m in data.get("data", [])]
        print(f"  [PASS    ] Model endpoint reachable")
        print(f"             Available models: {models}")
except urllib.error.HTTPError as e:
    print(f"  [FAIL    ] HTTP {e.code} — check model deployment or API key")
except Exception as e:
    print(f"  [FAIL    ] {e}")

print("")
print("IDE model verification:")
print("  Cursor:      Settings → Models → select model → test chat")
print("  VS Code:     Chat model picker → select RHOAI model → test")
print("  Claude Code: Launch with env vars → test response")

=== Model Endpoint Check ===

  [PASS    ] Model endpoint reachable
             Available models: ['qwen36-27b']

IDE model verification:
  Cursor:      Settings → Models → select model → test chat
  VS Code:     Chat model picker → select RHOAI model → test
  Claude Code: Launch with env vars → test response


---
## Summary

| IDE | Model Config | Launch |
|-----|-------------|--------|
| **Cursor** | Settings → Models → OpenAI Compatible | Normal launch |
| **VS Code** | `.vscode/chatLanguageModels.json` (BYOK) | Normal launch |
| **Claude Code** | Env vars (`ANTHROPIC_BASE_URL` + `ANTHROPIC_AUTH_TOKEN`) | `source run-claude.sh` |

> **MCP server registration** → `../1_mcp_servers/4_connect_ide_clients.ipynb`

### Key Points

- All IDEs connect to the **same self-hosted model** — no external API keys needed
- Claude Code uses `ANTHROPIC_AUTH_TOKEN` (not `ANTHROPIC_API_KEY`) for MaaS authentication
- MaaS Gateway (Phase 2) provides unified auth + rate limiting for all endpoints

## Next Steps

- `2_run_public_coding_assistant.ipynb` — Run with all 5 MCP tools (internet required)
- `3_run_closed_coding_assistant.ipynb` — Run with 3 local tools only (air-gapped)
- `../2_maas/2_enable_maas.ipynb` — Enable MaaS for production API key management